# Phase 3 & 4 — Stage 1: QLoRA SFT Training Loop
### Llama-3.2-3B SFT & Alignment Project

**Goal**: Train the raw `Llama-3.2-3B` base model using **QLoRA (4-bit NF4)** on our curated Stage 1 SFT dataset (`OpenHermes-2.5`, `Dolly-15k`, `GSM8K`, `MetaMathQA`, `HotpotQA`).

**What you will learn & execute:**
1. How **QLoRA** quantizes base weights into 4-bit NF4 while keeping trainable LoRA adapters in 16-bit.
2. How **Unsloth** accelerates training by 2-5x and cuts VRAM usage by 60% with custom CUDA kernels.
3. How to load your private dataset directly from HuggingFace Hub.
4. How to configure `SFTTrainer` with gradient accumulation, cosine LR decay, and grad norm clipping.
5. How to evaluate qualitative responses before vs. after training.
6. How to save adapters and push the Stage 1 model to HuggingFace Hub.

---
### Architecture Choice: Why Unsloth over Raw HuggingFace?

The reference project (Tan's notebook) uses the raw HuggingFace stack (`AutoModelForCausalLM` + `BitsAndBytesConfig` + `LoraConfig` manually). We use **Unsloth** instead because:
- **2-5x faster training** via custom hand-written CUDA kernels for attention and cross-entropy
- **~60% VRAM reduction** — makes T4 (15GB) a viable training target
- **Gradient checkpointing in `"unsloth"` mode** — more memory-efficient than the vanilla HF version
- **Drop-in compatibility** with `SFTTrainer`, `LoraConfig`, and HF Hub — no workflow change

This is not 'cheating' — Unsloth is a legitimate, production-grade optimization layer. The model it produces is identical to a vanilla QLoRA model.

> ⚠️ **GPU Required**: Make sure your Colab runtime is set to **GPU (T4, L4, or A100)**.

## 1. Setup & Environment Check

In [ ]:
# Install Unsloth with Colab-optimized builds and all required dependencies
# --no-deps prevents pip from pulling conflicting transitive dependencies
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes xformers huggingface_hub datasets

In [ ]:
# Verify GPU is available and get hardware specs
# This matters because:
#   - T4 (15GB): use per_device_batch_size=4 + grad_accum=8 to hit effective batch_size=32
#   - A100 (40/80GB): can use per_device_batch_size=32 directly
#   - bf16 support: A100/L4 support bfloat16; T4 only supports fp16
import torch
print(f"PyTorch Version : {torch.__version__}")
print(f"CUDA Available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device Name : {torch.cuda.get_device_name(0)}")
    total_vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM Available  : {total_vram:.2f} GB")
    print(f"bf16 Supported  : {torch.cuda.is_bf16_supported()}")
else:
    raise RuntimeError("No GPU detected! Go to Runtime → Change runtime type → GPU.")

In [ ]:
# Authenticate with HuggingFace Hub
# Required to:
#   1. Load our private SFT dataset (from Phase 2 pipeline)
#   2. Push our trained adapter back to Hub after training
from huggingface_hub import notebook_login
notebook_login()

## 2. Configuration & Hyperparameters

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURATION — single source of truth for all hyperparameters
# Change values HERE only — every subsequent cell reads from these variables
# ─────────────────────────────────────────────────────────────────────────────

# ── HuggingFace Identity ──────────────────────────────────────────────────────
HF_USERNAME       = "SpongeBob341"                          # <-- Your HF username
SFT_DATASET_REPO  = f"{HF_USERNAME}/llama3.2-3b-sft-dataset" # Phase 2 output repo
STAGE1_MODEL_REPO = f"{HF_USERNAME}/llama3.2-3b-sft-stage1" # Where adapter will be pushed

# ── Model Identity ────────────────────────────────────────────────────────────
# We use unsloth's pre-patched version of Llama-3.2-3B.
# 'unsloth/Llama-3.2-3B' = the same base weights as 'meta-llama/Llama-3.2-3B'
# but pre-patched with Unsloth's CUDA kernels. No quality difference.
MODEL_NAME     = "unsloth/Llama-3.2-3B"
MAX_SEQ_LENGTH = 512   # Token sequence limit. Locked in Phase 1 (O(n²) attention cost)
LOAD_IN_4BIT   = True  # 4-bit NF4 quantization. Reduces ~6GB model → ~1.8GB GPU footprint

# ── LoRA Adapter Configuration ───────────────────────────────────────────────
# Reference (Tan's notebook): r=64, alpha=16, dropout=0.1
# Our choice: r=16, alpha=16, dropout=0 — because:
#   - r=64 requires ~8GB extra VRAM for the adapter matrices. OOM risk on T4.
#   - r=16 is the standard T4-safe LoRA rank. Still injects 40M+ trainable params.
#   - dropout=0 is required by Unsloth: its custom kernels are incompatible with
#     non-zero LoRA dropout (using it produces an immediate warning and fallback).
LORA_R         = 16
LORA_ALPHA     = 16    # Scaling factor. Standard practice: alpha = r or 2*r
LORA_DROPOUT   = 0     # Must be 0 when using Unsloth (see rationale above)

# All 7 projection layers targeted — maximum expressiveness for the LoRA rank budget.
# q_proj, k_proj, v_proj, o_proj = attention mechanism
# gate_proj, up_proj, down_proj  = MLP (feed-forward) layers
# Tan's notebook uses the same 7 target modules.
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj",
                  "gate_proj", "up_proj", "down_proj"]

# ── Training Hyperparameters ─────────────────────────────────────────────────
# Effective batch size = PER_DEVICE_BATCH_SIZE × GRAD_ACCUM_STEPS = 4 × 8 = 32
# This matches Tan's batch_size=32 (which he ran directly on A100 with no accumulation).
# Gradient accumulation simulates a larger batch by accumulating gradients over N
# micro-steps before performing one weight update — same math, different memory profile.
PER_DEVICE_BATCH_SIZE = 4
GRAD_ACCUM_STEPS      = 8
LEARNING_RATE         = 2e-4    # Standard QLoRA SFT learning rate
NUM_EPOCHS            = 1       # Reference trained 1 epoch on 88k samples
WARMUP_RATIO          = 0.03    # First 3% of steps ramp LR from 0 → LEARNING_RATE
WEIGHT_DECAY          = 0.001   # L2 regularization (aligned with Tan's 0.001)

# max_grad_norm: clip gradients whose L2 norm exceeds this value.
# Prevents exploding gradients, especially important in QLoRA where base weights
# are frozen at 4-bit precision and adapter gradients can spike.
# Tan uses 0.3. Standard for QLoRA SFT.
MAX_GRAD_NORM         = 0.3

print(f"Config ready.")
print(f"  Dataset  : {SFT_DATASET_REPO}")
print(f"  Push to  : {STAGE1_MODEL_REPO}")
print(f"  Eff. Batch: {PER_DEVICE_BATCH_SIZE * GRAD_ACCUM_STEPS}")

## 3. Load Base Model & Attach LoRA Adapters via Unsloth

In [ ]:
from unsloth import FastLanguageModel

# Step 1: Load the base model in 4-bit NF4 quantization
# NF4 (Normal Float 4-bit) is optimal for normally distributed neural network weights.
# Unsloth handles the BitsAndBytesConfig internally — it patches the model with its
# own CUDA kernels for NF4 dequantization AND forward pass, giving the 2-5x speedup.
# dtype=None → Unsloth auto-detects: bfloat16 on A100/L4, float16 on T4
print("Loading base Llama-3.2-3B with Unsloth in 4-bit QLoRA...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,           # Auto-detect fp16/bf16 based on GPU capability
    load_in_4bit=LOAD_IN_4BIT,
)

# Step 2: Inject LoRA adapters into the target projection layers
# After this call:
#   - Base model weights: frozen, 4-bit NF4, not updated during training
#   - LoRA adapter matrices (A and B for each target layer): trainable, fp16/bf16
#   - Trainable params ≈ 40M out of 3.3B total (only ~2.9% of all params are trained)
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=TARGET_MODULES,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    # 'unsloth' gradient checkpointing: Unsloth's custom implementation that
    # uses less VRAM than vanilla HF gradient checkpointing by keeping fewer
    # intermediate activations in GPU memory during the backward pass.
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

print("\n✅ QLoRA Model Initialized!")
model.print_trainable_parameters()

# Also ensure pad_token is set — Llama base model has no pad token by default.
# We set it equal to eos_token (same as reference), which is standard for causal LM SFT.
# Right-side padding ensures the completion tokens are not masked during training.
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
print(f"Pad token: {tokenizer.pad_token} (id={tokenizer.pad_token_id})")

## 4. Load SFT Dataset from HuggingFace Hub

In [ ]:
from datasets import load_dataset

# Our Phase 2 pipeline pushed a pre-formatted dataset to HuggingFace Hub.
# Every sample has a single 'text' column in Alpaca format:
#
#   Instruction:
#   {instruction}
#
#   ### Input:
#   {optional_context}
#
#   ### Response:
#   {answer}<|end_of_text|>
#
# The dataset was pre-tokenized-length-filtered (max 512 tokens) and split 98/2 train/test.
# We load it directly — no additional processing needed here.
print(f"Downloading SFT dataset: {SFT_DATASET_REPO}")
sft_dataset = load_dataset(SFT_DATASET_REPO)

train_data = sft_dataset["train"]
eval_data  = sft_dataset["test"]

print(f"Train samples : {len(train_data):,}")
print(f"Eval samples  : {len(eval_data):,}")
print("\nSample preview (first 300 chars):")
print("-" * 60)
print(train_data[0]["text"][:300])
print("-" * 60)

## 5. Configure & Execute SFT Training Loop

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
import os

# Create output directories (trainer will fail if they don't exist)
OUTPUT_DIR  = "./sft_stage1_output"
LOGGING_DIR = "./sft_stage1_logs"
os.makedirs(OUTPUT_DIR,  exist_ok=True)
os.makedirs(LOGGING_DIR, exist_ok=True)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_data,
    eval_dataset=eval_data,

    # dataset_text_field: which column in our dataset contains the formatted text.
    # Our Phase 2 output uses the 'text' column — SFTTrainer will tokenize it internally.
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,

    # packing=False: we do NOT pack multiple short samples into one 512-token window.
    # Reason: our Phase 2 pipeline already filtered out samples >512 tokens,
    # and packing with an Alpaca-format dataset can cause the loss to bleed across
    # sample boundaries if not carefully masked. Simpler = more reliable.
    packing=False,

    args=TrainingArguments(
        output_dir=OUTPUT_DIR,
        logging_dir=LOGGING_DIR,

        # ── Batch & Gradient ──────────────────────────────────────────────────
        per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM_STEPS,  # Effective batch = 32

        # ── Learning Rate Schedule ────────────────────────────────────────────
        learning_rate=LEARNING_RATE,
        warmup_ratio=WARMUP_RATIO,         # Ramp LR over first 3% of steps
        lr_scheduler_type="cosine",        # Cosine decay after warmup
        num_train_epochs=NUM_EPOCHS,

        # ── Regularization ────────────────────────────────────────────────────
        weight_decay=WEIGHT_DECAY,

        # max_grad_norm: gradient clipping threshold.
        # If the L2 norm of gradients exceeds 0.3, they are rescaled back to 0.3.
        # Prevents rare but catastrophic gradient explosion events during QLoRA training.
        # Tan's reference uses 0.3. This is the standard for QLoRA SFT.
        max_grad_norm=MAX_GRAD_NORM,

        # ── Precision ─────────────────────────────────────────────────────────
        # bf16 on A100/L4 (numerically more stable), fp16 on T4 (no bf16 support)
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),

        # ── Optimizer ─────────────────────────────────────────────────────────
        optim="paged_adamw_32bit",  # Memory-efficient Adam for QLoRA
        # paged_adamw_32bit: stores optimizer states (momentum, variance) in
        # CPU RAM and 'pages' them to GPU only when needed. Saves ~6-8GB VRAM
        # vs. standard AdamW which keeps all optimizer states on GPU.

        # ── Evaluation & Saving ───────────────────────────────────────────────
        eval_strategy="steps",     # Run validation every N steps
        eval_steps=200,            # ~6% of training steps for a 35k dataset
        save_strategy="steps",
        save_steps=200,
        save_total_limit=2,        # Keep only the 2 most recent checkpoints
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,

        # ── Efficiency ────────────────────────────────────────────────────────
        # group_by_length: sorts samples into batches by similar token length.
        # Minimizes padding waste within each batch. Can speed up training by
        # 5-15% at no quality cost. Tan's reference uses this.
        group_by_length=True,

        # ── Logging ───────────────────────────────────────────────────────────
        logging_steps=10,
        report_to="none",   # Set to 'tensorboard' if you want loss curves

        seed=42,
    ),
)

print("Starting Stage 1 QLoRA SFT Training...")
trainer_stats = trainer.train()

print(f"\n✅ Stage 1 Training Complete!")
print(f"   Runtime   : {trainer_stats.metrics['train_runtime']:.2f} seconds")
print(f"   Train Loss: {trainer_stats.metrics['train_loss']:.4f}")
print(f"   Samples/s : {trainer_stats.metrics.get('train_samples_per_second', 'N/A')}")

## 6. Qualitative Sanity Check (Test Inference)

In [ ]:
# Switch model from training mode to optimized inference mode.
# FastLanguageModel.for_inference() disables dropout layers, enables KV-cache,
# and activates Unsloth's 2x faster inference kernels.
FastLanguageModel.for_inference(model)

# Test prompts use the SAME Alpaca format as training:
#   Instruction:\n{text}\n\n### Response:\n
# This is critical — if the format at inference doesn't match training, the model
# won't know when its response begins and will produce incoherent outputs.
test_prompts = [
    "Instruction:\nExplain the difference between supervised and unsupervised learning.\n\n### Response:\n",
    "Instruction:\nSolve the following math problem step by step.\nIf a train travels at 60 mph for 2.5 hours, how far does it travel?\n\n### Response:\n",
    "Instruction:\nWhat is the capital of France, and what is it known for?\n\n### Response:\n",
]

for i, prompt in enumerate(test_prompts):
    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.7,
        top_p=0.9,
        use_cache=True,   # KV-cache: reuses previously computed key/value pairs → faster
    )
    # Decode only the generated tokens (not the prompt)
    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    print(f"=== Test Prompt {i+1} ===")
    print(f"Prompt: {prompt.strip()[:80]}...")
    print(f"Response: {response}")
    print("-" * 60)

## 7. Save Adapter & Push Model to HuggingFace Hub

In [ ]:
# Save the LoRA adapter weights (NOT the full model — just the ~40M trainable params).
# The adapter directory will contain:
#   - adapter_config.json  ← LoRA architecture config (r, alpha, target_modules, etc.)
#   - adapter_model.bin    ← The actual trained adapter weights
#   - tokenizer files      ← Needed for loading later
#
# Why save adapters (not full merged model)?
#   - Adapter: ~160MB vs. merged model: ~6GB
#   - We can merge later (Phase 4.5) before deploying with vLLM
#   - Easier to iterate and resume training from an adapter checkpoint
LOCAL_SAVE_DIR = "./llama3.2-3b-sft-stage1-adapter"
model.save_pretrained(LOCAL_SAVE_DIR)
tokenizer.save_pretrained(LOCAL_SAVE_DIR)
print(f"✅ Adapter saved locally to: {LOCAL_SAVE_DIR}")

# Push to HuggingFace Hub (private repo)
print(f"\nPushing Stage 1 SFT Adapter to: {STAGE1_MODEL_REPO}")
model.push_to_hub(STAGE1_MODEL_REPO, private=True)
tokenizer.push_to_hub(STAGE1_MODEL_REPO, private=True)
print(f"\n✅ Stage 1 SFT Adapter pushed to: https://huggingface.co/{STAGE1_MODEL_REPO}")

## Phase 3 & 4 Complete

| Deliverable | Location | Status |
|---|---|---|
| **Stage 1 SFT Adapter** | `https://huggingface.co/{STAGE1_MODEL_REPO}` | ✅ Saved & Pushed |

### Training Config Summary

| Parameter | Value | Note |
|-----------|-------|------|
| `lora_r` | 16 | T4-safe (Tan used r=64 on A100) |
| `lora_alpha` | 16 | Standard scaling |
| `lora_dropout` | 0 | Required by Unsloth |
| `target_modules` | q,k,v,o,gate,up,down | Matches reference |
| `learning_rate` | 2e-4 | Matches reference |
| `effective_batch_size` | 32 | 4 × 8 grad accum |
| `max_grad_norm` | 0.3 | Matches reference |
| `group_by_length` | True | Matches reference |
| `num_train_epochs` | 1 | Matches reference |
| `lr_scheduler` | cosine | Matches reference |
| `optimizer` | paged_adamw_32bit | Matches reference |

### Next Steps
- **Phase 4.5**: Run `Phase4_5_Stage2_QLoRA_ORPO.ipynb` to align Stage 1 model using `ORPOTrainer` on UltraFeedback, HH-RLHF, and HaluEval.
- **Before Phase 4.5**: Merge Stage 1 adapter into the base model so ORPO has a clean starting point.